# Workshop self-test: *Swarm in the Lab*

**BiocAsia 2026 · satellite of ABACBS 2026 · RMIT University**

This notebook checks that your workshop machine is ready for the afternoon.

**How to run it**
1. In your remote desktop, open VS Code. It should already show the `workshop` folder.
2. Open this notebook. If VS Code asks for a kernel, choose **Workshop (Python 3.12)**.
3. If you were given API keys on paper or by email, run **Step 0** first.
   (If VS Code asks whether you trust the authors of this folder, choose **Yes**: it's the workshop folder.)
4. Click **Run All**. It takes about two minutes and costs less than one US cent.

**Reading the results:** ✅ ready · ⚠️ works, but worth a look · ❌ needs fixing · ⏭️ skipped

The last cell prints one line starting with `PREFLIGHT`. If it says **READY**, you're all set. If not, show it to a helper.

> **Keep your keys secret.** Workshop keys are only ever typed into the hidden box in Step 0. Never paste them into a notebook cell, a terminal or a Claude Code chat. This notebook never shows them: it shows only their length and a short fingerprint.

## Settings
*Participants: you don't need to change anything here. Organisers: fill in the `[PLACEHOLDER]` values once, before the machine image is made.*

In [ ]:
from pathlib import Path

WORKSHOP_DIR        = Path.home() / "workshop"
OUTPUT_DIR          = WORKSHOP_DIR / "outputs"
DATASET_PATH        = WORKSHOP_DIR / "data" / "workshop.h5ad"          # Module 2 dataset (tools/make_workshop_dataset.py)
PAPER2AGENT_DIR     = WORKSHOP_DIR / "Paper2Agent"
VIRTUAL_BIOTECH_DIR = WORKSHOP_DIR / "virtual-biotech"
REQUIRED_NOTEBOOKS  = ["01_virtual_lab.ipynb", "02_paper2agent.ipynb", "03_virtual_biotech.ipynb"]
WORKSHOP_ENV        = "swarm-lab"                                      # name of the conda environment
CONDA_BASE          = Path.home() / "miniforge3"
VBT_ENV             = "vbt"                                            # the Virtual Biotech's own environment (Python 3.11)
OPEN_TARGETS_DATA   = WORKSHOP_DIR / "data" / "open_targets"           # Open Targets 25.09 download

TARGET_GENE_SYMBOL = "CD276"            # B7-H3
TARGET_ENSEMBL_ID  = "ENSG00000103855"
TRIAL_TERM         = "B7-H3 OR CD276"
TRIAL_CONDITION    = "lung cancer"

OPENAI_MODEL = "gpt-5.2"   # the model Virtual Lab uses in Module 1
CLAUDE_MODEL = "haiku"     # cheap Agent SDK check
VBT_MODEL    = "claude-sonnet-5-5"   # must match MODEL in 03_virtual_biotech.ipynb

# Pre-built Paper2Agent MCP servers: name -> {"command": ..., "args": [...]} (local) or an https URL. {} skips the check.
PAPER_AGENTS_DIR = WORKSHOP_DIR / "paper_agents"
PAPER_AGENT_SERVERS = {
    "scanpy": {"command": str(PAPER_AGENTS_DIR / "scanpy-mcp" / ".venv" / "bin" / "python"),
               "args": [str(PAPER_AGENTS_DIR / "scanpy-mcp" / "src" / "scanpy_mcp_stdio.py")]},
    "tissue": {"command": str(PAPER_AGENTS_DIR / "TISSUE-mcp" / ".venv" / "bin" / "python"),
               "args": [str(PAPER_AGENTS_DIR / "TISSUE-mcp" / "src" / "tissue_mcp_stdio.py")]},
}

# Network ports that may accept connections from outside this machine. Anything else is flagged.
EXPECTED_PORTS = {22: "SSH", 8443: "Amazon DCV", 5355: "name lookup (systemd-resolved)"}

TIMEOUT_S = 30

In [ ]:
# Helper functions. No need to read this cell.
import asyncio, hashlib, json, os, platform, re, shutil, socket, subprocess, sys, tempfile, time
import urllib.error, urllib.parse, urllib.request
from importlib import metadata

# --- Hiding secrets ------------------------------------------------------------------
# Anything that looks like a key or token is replaced before it's shown on screen or saved.
_SECRET_VALUES = set()
_SECRET_RE = re.compile(   # (?<!…) stops ordinary words like "risk-assessment" from matching
    r"(?<![A-Za-z0-9])(?:sk-[A-Za-z0-9_*.\-]{16,}|hf_[A-Za-z0-9]{20,}|gh[pousr]_[A-Za-z0-9]{20,}"
    r"|github_pat_[A-Za-z0-9_]{20,}|(?:AKIA|ASIA)[A-Z0-9]{16}\b"
    r"|eyJ[A-Za-z0-9_\-]{10,}\.[A-Za-z0-9_\-]{10,}\.[A-Za-z0-9_\-]{10,})")

def redact(text):
    s = "" if text is None else str(text)
    for value in _SECRET_VALUES:
        s = s.replace(value, "[hidden]")
    return _SECRET_RE.sub("[hidden]", s)

# --- Workshop keys --------------------------------------------------------------------
# Keys live in one file that only your user can read. They are loaded into this notebook's
# process only: not into your terminal, and not into Claude Code. They are never printed.
KEY_FILE = Path.home() / ".config" / "swarm-lab" / "keys.env"
KEY_NAMES = ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "OPENAI_BASE_URL", "ANTHROPIC_BASE_URL")

def _read_key_file():
    values = {}
    try:
        lines = KEY_FILE.read_text().splitlines()
    except OSError:
        return values
    for line in lines:
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        name, value = line.split("=", 1)
        name = name.strip().removeprefix("export ").strip()
        value = value.strip().strip('"').strip("'")
        if name in KEY_NAMES and value:
            values[name] = value
    return values

def key_file_is_private():
    """True if only the owner can read the key file and its folder."""
    try:
        return (KEY_FILE.stat().st_mode & 0o077) == 0 and (KEY_FILE.parent.stat().st_mode & 0o077) == 0
    except OSError:
        return False

def load_keys(names):
    """Load the named keys into this notebook only. Returns where each came from:
    'environment', 'key file' or None (missing)."""
    from_file = _read_key_file()
    sources = {}
    for name in names:
        if os.environ.get(name):
            sources[name] = "environment"
        elif name in from_file:
            os.environ[name] = from_file[name]
            sources[name] = "key file"
        else:
            sources[name] = None
        if name.endswith("_KEY") and os.environ.get(name):
            _SECRET_VALUES.add(os.environ[name])
    return sources

def fingerprint(value):
    """A short label that tells keys apart without revealing them."""
    return hashlib.sha256(value.encode()).hexdigest()[:8]

RESULTS = []
ICONS = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌", "SKIP": "⏭️"}

def record(name, status, detail="", fix=""):
    RESULTS[:] = [r for r in RESULTS if r["check"] != name]
    detail, fix = redact(detail), redact(fix if status in ("WARN", "FAIL") else "")
    RESULTS.append({"check": name, "status": status, "detail": detail, "fix": fix})
    print(f"{ICONS[status]} {name}: {detail}")
    if fix:
        print(f"      ↳ {fix}")

def _describe(e):
    text = str(e).strip().splitlines()[0][:200] if str(e).strip() else ""
    return f"{type(e).__name__}: {text}"

MISSING_PKG = "A package is missing from this kernel. Check that the kernel is 'Workshop (Python 3.12)'; if it is, tell a helper."

def _hint(fix, e):
    if isinstance(e, ImportError):
        return MISSING_PKG
    return fix(e) if callable(fix) else fix

def check(name, fn, fix="", on_error="FAIL"):
    try:
        status, detail = fn()
        hint = fix if isinstance(fix, str) else ""
    except Exception as e:
        status, detail = ("FAIL" if isinstance(e, ImportError) else on_error), _describe(e)
        hint = _hint(fix, e)
    record(name, status, detail, hint)

async def acheck(name, coro_fn, fix="", timeout=90, on_error="FAIL"):
    try:
        status, detail = await asyncio.wait_for(coro_fn(), timeout)
        hint = fix if isinstance(fix, str) else ""
    except asyncio.TimeoutError:
        status, detail, hint = on_error, f"no answer after {timeout}s", _hint(fix, None)
    except Exception as e:
        status, detail = ("FAIL" if isinstance(e, ImportError) else on_error), _describe(e)
        hint = _hint(fix, e)
    record(name, status, detail, hint)

def http_json(url, params=None, payload=None, timeout=TIMEOUT_S):
    if not url.startswith("https://"):
        raise ValueError("refusing a non-HTTPS address")   # every call leaves this machine encrypted
    if params:
        url += "?" + urllib.parse.urlencode(params)
    data = json.dumps(payload).encode() if payload is not None else None
    headers = {"User-Agent": "workshop-preflight/1.0", "Accept": "application/json"}
    if data:
        headers["Content-Type"] = "application/json"
    with urllib.request.urlopen(urllib.request.Request(url, data=data, headers=headers), timeout=timeout) as r:
        return json.loads(r.read().decode())

def api_hint(e):
    n = type(e).__name__ if e is not None else "Timeout"
    if "Authentication" in n or "PermissionDenied" in n:
        return "The key was rejected. Set REENTER = True in Step 0, run Step 0, then run this cell again; or tell a helper."
    if n == "NotFoundError":
        return "The model name wasn't recognised. Organisers: check the Settings cell."
    if "RateLimit" in n:
        return "Rate-limited, or this machine's spending cap is used up. Wait a minute and re-run; then tell a helper."
    if "Connection" in n or "Timeout" in n:
        return "Couldn't reach the API. Re-run this cell; if it keeps failing, tell a helper."
    return "Re-run this cell once. If it fails again, tell a helper."

HELPER = "Tell a helper. This is a machine setup problem, not something you did."
print("Helpers loaded.")

## Step 0 · Your workshop keys
*Skip this if the machine came with keys already set up. The cell below says so.*

Each key is pasted into a **hidden box** (VS Code shows it at the top of the window) and saved to a file only your user can read. It is never written into this notebook. If a key was rejected later on, set `REENTER = True` and run this cell again.

In [ ]:
import getpass

REENTER = False   # True = type the keys again, e.g. if one was rejected
saved = _read_key_file()
missing = [n for n in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY")
           if REENTER or (not os.environ.get(n) and n not in saved)]
if not missing:
    print(f"✅ Workshop keys are already set up in {KEY_FILE}. Nothing to do.")
else:
    for name in missing:
        entered = getpass.getpass(f"Paste your {name} (hidden), or press Enter to skip: ").strip()
        if entered:
            saved[name] = entered
    KEY_FILE.parent.mkdir(parents=True, exist_ok=True)
    os.chmod(KEY_FILE.parent, 0o700)
    fd = os.open(KEY_FILE, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    with os.fdopen(fd, "w") as f:
        f.write("# Workshop API keys. Owner-only. Never copy this file into notebooks, chats or git.\n")
        for name, value in saved.items():
            f.write(f"{name}={value}\n")
    entered = None
    print(f"✅ Saved {sum(n in saved for n in missing)} key(s) to {KEY_FILE} (readable only by you).")
if KEY_FILE.exists():
    os.chmod(KEY_FILE.parent, 0o700)
    os.chmod(KEY_FILE, 0o600)
del saved

## 1 · The machine

In [ ]:
def _python():
    v = sys.version_info
    in_env = Path(sys.prefix).name == WORKSHOP_ENV
    detail = f"Python {platform.python_version()} in {sys.prefix}"
    if (3, 12) <= v[:2] < (3, 13) and in_env:
        return "PASS", detail
    return "FAIL", detail
check("Python 3.12 in the workshop environment", _python,
      fix="Choose the kernel 'Workshop (Python 3.12)' in the top-right corner, then Run All again. "
          "(Virtual Lab needs 3.12 or later; CELLxGENE Census supports up to 3.12.)")

def _resources():
    cpus = os.cpu_count() or 0
    mem_gb = 0.0
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemTotal:"):
                    mem_gb = int(line.split()[1]) / 1024**2
    except OSError:
        pass
    free_gb = shutil.disk_usage(Path.home()).free / 1024**3
    ok = cpus >= 4 and mem_gb >= 14 and free_gb >= 5
    return ("PASS" if ok else "WARN"), f"{cpus} vCPU · {mem_gb:.1f} GB RAM · {free_gb:.1f} GB free in your home folder"
check("CPU, memory and disk", _resources,
      fix="Smaller than expected (4 vCPU, 16 GB RAM, 5+ GB free). It may still work; mention it to a helper.")

def _terminal_python():
    # Claude Code runs commands in a new terminal, so that terminal must use the workshop Python too.
    env = {"HOME": str(Path.home()), "USER": os.environ.get("USER", ""), "TERM": "dumb",
           "PATH": "/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin"}
    r = subprocess.run(["bash", "-ic", "python -c 'import sys; print(sys.prefix)' 2>/dev/null || echo NO_PYTHON"],
                       capture_output=True, text=True, timeout=30, env=env, stdin=subprocess.DEVNULL)
    last = (r.stdout.strip().splitlines() or ["NO_PYTHON"])[-1].strip()
    if last == "NO_PYTHON":
        return "WARN", "a new terminal has no `python` command"
    if Path(last).resolve() == Path(sys.prefix).resolve():
        return "PASS", "new terminals use the workshop environment"
    return "WARN", f"new terminals use {last}"
check("Terminals use the workshop Python", _terminal_python,
      fix="Claude Code would run the wrong Python. Organisers: add `conda activate swarm-lab` to ~/.bashrc.")

## 2 · Software on the machine

In [ ]:
PACKAGES = [  # (label, import name, distribution name)
    ("virtual-lab (Module 1)",      "virtual_lab",      "virtual-lab"),
    ("openai",                      "openai",           "openai"),
    ("claude-agent-sdk (Module 3)", "claude_agent_sdk", "claude-agent-sdk"),
    ("fastmcp (Modules 2-3)",       "fastmcp",          "fastmcp"),
    ("scanpy",                      "scanpy",           "scanpy"),
    ("anndata",                     "anndata",          "anndata"),
    ("cellxgene-census",            "cellxgene_census", "cellxgene-census"),
    ("pandas",                      "pandas",           "pandas"),
    ("biopython",                   "Bio",              "biopython"),
]

def _pkg(mod, dist):
    def fn():
        __import__(mod)
        try:
            return "PASS", metadata.version(dist)
        except metadata.PackageNotFoundError:
            return "PASS", "version unknown"
    return fn

for label, mod, dist in PACKAGES:
    check(label, _pkg(mod, dist))

def _folders():
    missing = [str(p) for p in (WORKSHOP_DIR, PAPER2AGENT_DIR, VIRTUAL_BIOTECH_DIR) if not Path(p).exists()]
    missing += [n for n in REQUIRED_NOTEBOOKS if not (WORKSHOP_DIR / n).exists()]
    if missing:
        return "FAIL", "missing: " + ", ".join(missing)
    return "PASS", f"workshop folder, Paper2Agent, Virtual Biotech and {len(REQUIRED_NOTEBOOKS)} module notebooks found"
check("Workshop materials", _folders, fix=HELPER)

def _virtual_biotech():
    problems = []
    if not (CONDA_BASE / "envs" / VBT_ENV / "bin" / "python").exists():
        problems.append(f"no '{VBT_ENV}' conda environment")
    if not (VIRTUAL_BIOTECH_DIR / "mcp_config.json").exists():
        problems.append("mcp_config.json missing (run setup_mcp.py)")
    if not (WORKSHOP_DIR / "tools" / "vbt_workshop.py").exists():
        problems.append("tools/vbt_workshop.py missing")
    # The dataset list the Virtual Biotech requires, read as text from its config (no code is run).
    try:
        names = re.findall(r"'([a-z_0-9]+)'", (VIRTUAL_BIOTECH_DIR / "src" / "config" / "datasets.py").read_text())
    except OSError:
        names = []
    missing = [n for n in names if not (OPEN_TARGETS_DATA / n).is_dir()]
    if not names:
        problems.append("can't read its dataset list")
    elif missing:
        problems.append(f"{len(missing)} of {len(names)} Open Targets datasets missing from {OPEN_TARGETS_DATA}")
    if problems:
        return "FAIL", "; ".join(problems)
    return "PASS", f"environment, tool servers and all {len(names)} Open Targets datasets present"
check("Virtual Biotech ready (Module 3)", _virtual_biotech, fix=HELPER)

## 3 · Security
These checks make sure keys stay private and nothing on the machine is open to the network that shouldn't be.

In [ ]:
def _key_file():
    if not KEY_FILE.exists():
        return "FAIL", f"no key file at {KEY_FILE}"
    if not key_file_is_private():
        return "FAIL", "the key file can be read by other users"
    return "PASS", "present and readable only by you"
check("Key file is private", _key_file, fix="Run Step 0 again: it creates the file and tightens its permissions.")

SOURCES = load_keys(["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "OPENAI_BASE_URL", "ANTHROPIC_BASE_URL"])

def _key(name, base):
    def fn():
        v = os.environ.get(name, "")
        if not v:
            return "FAIL", "missing"
        via = f", via gateway {os.environ[base]}" if os.environ.get(base) else ""
        detail = f"loaded from {SOURCES[name]} ({len(v)} characters, fingerprint {fingerprint(v)}{via})"
        if name == "ANTHROPIC_API_KEY" and SOURCES[name] == "environment":
            return "WARN", detail
        return "PASS", detail
    return fn
check("OpenAI key", _key("OPENAI_API_KEY", "OPENAI_BASE_URL"), fix="Run Step 0 to enter it.")
check("Anthropic key", _key("ANTHROPIC_API_KEY", "ANTHROPIC_BASE_URL"),
      fix="Run Step 0 to enter it. If it says 'environment': the key is set system-wide, so Claude Code in VS Code "
          "may use it instead of your own sign-in. Organisers: keep it only in the key file.")

def _startup_files():
    # Keys must not be exported by shell or conda start-up scripts: then every terminal, and Claude Code, would see them.
    files = [Path.home() / f for f in (".bashrc", ".profile", ".bash_profile", ".zshrc", ".zprofile")]
    files += [Path("/etc/environment"), Path("/etc/bash.bashrc")] + sorted(Path("/etc/profile.d").glob("*"))
    conda_etc = Path(sys.prefix) / "etc" / "conda"
    if conda_etc.is_dir():
        files += sorted(p for p in conda_etc.rglob("*") if p.is_file())
    files.append(Path(sys.prefix) / "conda-meta" / "state")
    files.append(VIRTUAL_BIOTECH_DIR / ".env")   # the Virtual Biotech's README suggests a key here; we don't use it
    pattern = re.compile(r"(OPENAI_API_KEY|ANTHROPIC_API_KEY|ANTHROPIC_AUTH_TOKEN|CLAUDE_CODE_OAUTH_TOKEN)\"?\s*[=:]")
    hits = []
    for f in files:
        try:
            text = f.read_text(errors="ignore")
        except OSError:
            continue
        if pattern.search(text) or _SECRET_RE.search(text):
            hits.append(str(f))
    if hits:
        return "WARN", "keys appear in: " + ", ".join(hits)
    return "PASS", "no keys in shell or conda start-up files"
check("Keys stay out of start-up files", _startup_files,
      fix="Organisers: remove keys from those files and keep them only in the key file.")

def _claude_settings():
    p = WORKSHOP_DIR / ".claude" / "settings.json"
    try:
        s = json.loads(p.read_text())
    except (OSError, ValueError):
        return "WARN", f"no readable {p}"
    perms = s.get("permissions") or {}
    deny = " ".join(perms.get("deny", []))
    problems = []
    if ".config/swarm-lab" not in deny:
        problems.append("the key file isn't blocked")
    if perms.get("disableBypassPermissionsMode") != "disable":
        problems.append("'bypass permissions' mode isn't disabled")
    return ("WARN", "; ".join(problems)) if problems else (
        "PASS", "Claude Code's file tools and file commands (cat, head…) are blocked from the key file, and its "
                "permission prompts can't be switched off")
check("Claude Code safety settings", _claude_settings,
      fix="Organisers: copy claude_settings.json to ~/workshop/.claude/settings.json.")

def _listening():
    owners = {}
    for pid in filter(str.isdigit, os.listdir("/proc")):
        try:
            name = Path(f"/proc/{pid}/comm").read_text().strip()
            for fd in os.listdir(f"/proc/{pid}/fd"):
                link = os.readlink(f"/proc/{pid}/fd/{fd}")
                if link.startswith("socket:["):
                    owners[link[8:-1]] = name
        except OSError:
            continue
    exposed = []
    for path in ("/proc/net/tcp", "/proc/net/tcp6"):
        try:
            lines = Path(path).read_text().splitlines()[1:]
        except OSError:
            continue
        for line in lines:
            parts = line.split()
            if len(parts) < 10 or parts[3] != "0A":       # 0A = listening
                continue
            addr, port_hex = parts[1].split(":")
            port = int(port_hex, 16)
            loopback = addr.endswith("7F") or addr == "00000000000000000000000001000000"
            if not loopback and port not in EXPECTED_PORTS:
                exposed.append(f"port {port} ({owners.get(parts[9], 'unknown program')})")
    if exposed:
        return "WARN", "open to the network: " + ", ".join(sorted(set(exposed)))
    return "PASS", "only " + ", ".join(f"{v} ({k})" for k, v in EXPECTED_PORTS.items()) + " can accept outside connections"
check("Nothing unexpected open to the network", _listening,
      fix="Something here accepts connections from other machines. If it's Jupyter, run notebooks inside VS Code "
          "instead (or start Jupyter with --ip=127.0.0.1). Otherwise tell a helper.")

## 4 · Hello, agent
**Virtual Lab** uses OpenAI and **Virtual Biotech** uses Anthropic, through the Claude Agent SDK. Each check sends one tiny message.

In [ ]:
def _vlab_objects():
    from virtual_lab.agent import Agent
    from virtual_lab.run_meeting import run_meeting  # noqa: F401
    pi = Agent(title="Principal Investigator", expertise="tumour immunology",
               goal="check the workshop setup", role="lead the meeting", model=OPENAI_MODEL)
    return "PASS", f"created a test agent ({getattr(pi, 'title', 'PI')}, {OPENAI_MODEL}); run_meeting is available"
check("Virtual Lab agents can be created", _vlab_objects, fix=HELPER)

def _openai_hello():
    from openai import OpenAI
    t0 = time.time()
    r = OpenAI(timeout=TIMEOUT_S * 2, max_retries=1).chat.completions.create(
        model=OPENAI_MODEL, messages=[{"role": "user", "content": "Reply with exactly two words: hello agent"}])
    text = (r.choices[0].message.content or "").strip()
    status = "PASS" if "hello" in text.lower() else "WARN"
    return status, f'{r.model} replied "{text[:40]}" in {time.time() - t0:.1f}s ({getattr(r.usage, "total_tokens", "?")} tokens)'
check("OpenAI says hello (Virtual Lab path)", _openai_hello, fix=api_hint)

def _vbt_model():
    # Is Module 3's model available to this key? A free lookup in Anthropic's model list: no model call.
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        return "SKIP", "no Anthropic key"
    base = os.environ.get("ANTHROPIC_BASE_URL", "https://api.anthropic.com").rstrip("/")
    if not base.startswith("https://"):
        raise ValueError("refusing a non-HTTPS address")
    req = urllib.request.Request(f"{base}/v1/models/{VBT_MODEL}", headers={
        "x-api-key": key, "anthropic-version": "2023-06-01", "User-Agent": "workshop-preflight/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=TIMEOUT_S) as r:
            d = json.loads(r.read().decode())
    except urllib.error.HTTPError as e:
        if e.code == 404:
            return "FAIL", f"{VBT_MODEL} isn't available to this key"
        raise
    return "PASS", f"{d.get('id', VBT_MODEL)} is available ({d.get('display_name', '')})"
check("Module 3's model is available", _vbt_model, on_error="WARN",
      fix="Organisers: set MODEL in 03_virtual_biotech.ipynb (and VBT_MODEL here) to a model these keys can use.")

In [ ]:
async def _claude_hello():
    from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock, ResultMessage
    opts = ClaudeAgentOptions(model=CLAUDE_MODEL, max_turns=1, cwd=tempfile.mkdtemp(prefix="preflight-"))
    t0, text, model, cost, error = time.time(), "", None, None, None
    async for m in query(prompt="Reply with exactly two words: hello agent", options=opts):
        if isinstance(m, AssistantMessage):
            model = getattr(m, "model", None) or model
            text += "".join(b.text for b in m.content if isinstance(b, TextBlock))
        elif isinstance(m, ResultMessage):
            cost = getattr(m, "total_cost_usd", None)
            if getattr(m, "is_error", False):
                error = f"{m.subtype}: {str(getattr(m, 'result', ''))[:150]}"
    if error:
        return "FAIL", error
    cost_s = f", about US${cost:.4f}" if isinstance(cost, (int, float)) else ""
    return ("PASS" if "hello" in text.lower() else "WARN"), f'{model or CLAUDE_MODEL} replied "{text.strip()[:40]}" in {time.time() - t0:.1f}s{cost_s}'

await acheck("Claude says hello (Agent SDK, Virtual Biotech path)", _claude_hello, fix=api_hint, timeout=120)

## 5 · Tools over MCP

In [ ]:
async def _mcp_local():
    from fastmcp import FastMCP, Client
    server = FastMCP("preflight")

    @server.tool()
    def echo_gene(symbol: str) -> str:
        """Return the gene symbol: a do-nothing tool for the self-test."""
        return f"{symbol} ok"

    async with Client(server) as c:
        tools = await c.list_tools()
        res = await c.call_tool("echo_gene", {"symbol": TARGET_GENE_SYMBOL})
    out = getattr(res, "data", None)
    if out is None:
        out = getattr(res, "content", res)[0].text
    return "PASS", f"local MCP server listed {len(tools)} tool and returned '{out}'"
await acheck("MCP works on this machine", _mcp_local, fix=HELPER, timeout=60)

def _transport(cfg):
    from fastmcp.client.transports import StdioTransport, StreamableHttpTransport
    if isinstance(cfg, str):
        if not cfg.startswith("https://"):
            raise ValueError("refusing a non-HTTPS server address")
        return StreamableHttpTransport(cfg)
    return StdioTransport(cfg["command"], cfg.get("args", []))

async def _mcp_remote(cfg):
    from fastmcp import Client
    async with Client(_transport(cfg)) as c:
        tools = await c.list_tools()
    names = ", ".join(t.name for t in tools[:4]) + ("…" if len(tools) > 4 else "")
    return ("PASS" if tools else "WARN"), f"{len(tools)} tools ({names})"

if not PAPER_AGENT_SERVERS:
    record("Paper2Agent servers (Module 2)", "SKIP", "none listed in Settings")
for name, cfg in PAPER_AGENT_SERVERS.items():
    await acheck(f"Paper agent: {name}", lambda c=cfg: _mcp_remote(c), timeout=180,
                 fix="Couldn't start this paper agent. Re-run this cell once, then tell a helper.")

## 6 · Public databases
All requests use HTTPS. Each asks one small question about the workshop target.

In [ ]:
NET_FIX = "Couldn't reach this database. Re-run this cell; if it keeps failing, tell a helper (pre-run results exist)."

def _opentargets():
    q = "query($id: String!) { target(ensemblId: $id) { approvedSymbol approvedName } }"
    d = http_json("https://api.platform.opentargets.org/api/v4/graphql",
                  payload={"query": q, "variables": {"id": TARGET_ENSEMBL_ID}})
    t = (d.get("data") or {}).get("target")
    return ("PASS", f"{t['approvedSymbol']}: {t['approvedName']}") if t else ("FAIL", "no answer for the target")
check("Open Targets", _opentargets, fix=NET_FIX)

def _ctgov():
    d = http_json("https://clinicaltrials.gov/api/v2/studies",
                  params={"query.term": TRIAL_TERM, "query.cond": TRIAL_CONDITION,
                          "countTotal": "true", "pageSize": 1, "fields": "NCTId"})
    return "PASS", f"{d.get('totalCount', '?')} studies match '{TRIAL_TERM}' in {TRIAL_CONDITION}"
check("ClinicalTrials.gov", _ctgov, fix=NET_FIX)

def _pubmed():
    d = http_json("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
                  params={"db": "pubmed", "term": f"{TARGET_GENE_SYMBOL} AND {TRIAL_CONDITION}",
                          "retmode": "json", "retmax": 0, "tool": "workshop_preflight"})
    return "PASS", f"{d['esearchresult']['count']} PubMed records"
check("PubMed (E-utilities)", _pubmed, fix=NET_FIX, on_error="WARN")

def _chembl():
    d = http_json("https://www.ebi.ac.uk/chembl/api/data/target/search.json", params={"q": TARGET_GENE_SYMBOL, "limit": 1})
    return "PASS", f"{d.get('page_meta', {}).get('total_count', '?')} ChEMBL targets match {TARGET_GENE_SYMBOL}"
check("ChEMBL", _chembl, fix=NET_FIX, on_error="WARN")

def _census():
    import cellxgene_census
    desc = cellxgene_census.get_census_version_description("stable")
    with cellxgene_census.open_soma(census_version="stable") as census:
        s = census["census_info"]["summary"].read().concat().to_pandas()
    total = s.set_index("label").loc["total_cell_count", "value"]
    return "PASS", f"Census {desc.get('release_build', 'stable')} opened ({int(total):,} cells)"
check("CELLxGENE Census", _census, fix=NET_FIX)

## 7 · Workshop data

In [ ]:
def _dataset():
    import anndata as ad
    a = ad.read_h5ad(DATASET_PATH, backed="r")
    try:
        names = set(map(str, a.var_names))
        for col in ("feature_name", "gene_symbols", "gene_symbol", "symbol"):
            if col in a.var.columns:
                names |= set(map(str, a.var[col]))
        found = TARGET_GENE_SYMBOL in names or any(n.split(".")[0] == TARGET_ENSEMBL_ID for n in names)
        detail = f"{a.n_obs:,} cells × {a.n_vars:,} genes; {TARGET_GENE_SYMBOL} {'is present' if found else 'NOT found'}"
    finally:
        a.file.close()
    return ("PASS" if found else "WARN"), detail
check("Workshop dataset loads", _dataset, fix=HELPER)

## Result

In [ ]:
counts = {s: sum(r["status"] == s for r in RESULTS) for s in ICONS}
ready = counts["FAIL"] == 0
line = (f"PREFLIGHT {'READY' if ready else 'NOT READY'} | {counts['PASS']} pass · {counts['WARN']} warn · "
        f"{counts['FAIL']} fail · {counts['SKIP']} skipped | {socket.gethostname()} | {time.strftime('%Y-%m-%d %H:%M')}")

attention = [r for r in RESULTS if r["status"] in ("FAIL", "WARN")]
if attention:
    print("Needs a look:\n")
    for r in sorted(attention, key=lambda r: r["status"] != "FAIL"):
        print(f"  {ICONS[r['status']]} {r['check']}: {r['detail']}")
        if r["fix"]:
            print(f"      ↳ {r['fix']}")
    print()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "preflight_result.json").write_text(redact(json.dumps({"summary": line, "results": RESULTS}, indent=2)))

bar = "=" * len(line)
print(bar, line, bar, sep="\n")
print("\nYou're all set. See you at the workshop!" if ready else "\nPlease show the PREFLIGHT line above to a helper.")

---
**For organisers**
- Run this notebook on your own instance before the image is made, and again on one clone afterwards.
- Before the image is made, run `prepare_image.py` (see the facilitators' setup guide). It removes your keys, sign-ins and this notebook's saved outputs.